# 🔬 Activity 06: Regularization — Ridge & Lasso with K-Fold Tuning

**Track:** Linear Regression  
**Level:** Advanced  
**K-Fold:** ✅ Used to tune alpha parameter

---

## 📖 What You Will Learn
- **Why regularisation** — overfitting and the curse of dimensionality
- **Ridge (L2)** — penalises large coefficients; shrinks towards zero
- **Lasso (L1)** — drives some coefficients to exactly zero (feature selection)
- **Elastic Net** — combines L1 + L2
- Use **K-Fold** to tune the `alpha` hyperparameter
- The **regularisation path** — how coefficients change with alpha

---

## 🧠 Concept: The Regularisation Penalty

**Ridge (L2):**
$$\mathcal{L}_{Ridge} = \underbrace{\|y - X\beta\|^2}_{\text{MSE}} + \underbrace{\alpha \|\beta\|^2}_{\text{L2 penalty}}$$

**Lasso (L1):**
$$\mathcal{L}_{Lasso} = \|y - X\beta\|^2 + \alpha \|\beta\|_1$$

**Key difference:**
- L2 → all coefficients shrink proportionally (never exactly 0)
- L1 → creates **sparsity** (some coefficients become exactly 0) → automatic feature selection

> **Rule of thumb:**  
> Many features, most relevant → Ridge  
> Many features, few relevant → Lasso  
> Unsure → Elastic Net

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV
)
from sklearn.model_selection import (
    KFold, cross_val_score, GridSearchCV, train_test_split
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Ready')

## 📊 Setup: Overfit Problem (Polynomial Features)

In [ ]:
housing = fetch_california_housing(as_frame=True)
X = housing.frame.drop(columns=['MedHouseVal'])
y = housing.frame['MedHouseVal']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Degree-2 polynomial features create collinearity and overfitting potential
poly_scaler = Pipeline([
    ('scaler', StandardScaler()),
    ('poly',   PolynomialFeatures(degree=2, include_bias=False))
])

X_tr_poly = poly_scaler.fit_transform(X_tr)
X_te_poly = poly_scaler.transform(X_te)

print(f'Original features: {X_tr.shape[1]}')
print(f'After poly(2):     {X_tr_poly.shape[1]}')

In [ ]:
# ── Baseline: plain OLS overfits on polynomial features ───────────────────────
lr = LinearRegression()
lr.fit(X_tr_poly, y_tr)

r2_train = r2_score(y_tr, lr.predict(X_tr_poly))
r2_test  = r2_score(y_te, lr.predict(X_te_poly))

print(f'OLS — Train R²: {r2_train:.4f}  Test R²: {r2_test:.4f}')
print(f'Gap (overfit indicator): {r2_train - r2_test:.4f}')
print(f'\nMax |coefficient|: {np.abs(lr.coef_).max():.1f}')

## 🔵 Section 1 — Ridge Regression

In [ ]:
# ── Ridge with K-Fold tuning ──────────────────────────────────────────────────
# WHY use RidgeCV? It uses an efficient leave-one-out CV internally (no need to loop).
# For Lasso, LassoCV iterates over alpha values across K-Fold splits.

alphas = np.logspace(-3, 4, 50)    # 50 values from 0.001 to 10,000
kf     = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_cv = RidgeCV(alphas=alphas, cv=kf)
ridge_cv.fit(X_tr_poly, y_tr)

best_alpha_ridge = ridge_cv.alpha_
print(f'Ridge best alpha (from 5-Fold CV): {best_alpha_ridge:.4f}')

r2_ridge_train = r2_score(y_tr, ridge_cv.predict(X_tr_poly))
r2_ridge_test  = r2_score(y_te, ridge_cv.predict(X_te_poly))

print(f'Ridge — Train R²: {r2_ridge_train:.4f}  Test R²: {r2_ridge_test:.4f}')
print(f'Gap: {r2_ridge_train - r2_ridge_test:.4f}  (reduced from OLS: {r2_train - r2_test:.4f})')

In [ ]:
# ── Manual alpha sweep — showing K-Fold stability ────────────────────────────
test_alphas = np.logspace(-2, 3, 30)
cv_means, cv_stds = [], []

for a in test_alphas:
    ridge = Ridge(alpha=a)
    scores = cross_val_score(ridge, X_tr_poly, y_tr, cv=kf, scoring='r2')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds  = np.array(cv_stds)

plt.figure(figsize=(10, 5))
plt.semilogx(test_alphas, cv_means, 'b-o', markersize=4, label='CV Mean R²')
plt.fill_between(test_alphas,
                  cv_means - cv_stds,
                  cv_means + cv_stds,
                  alpha=0.2, color='blue', label='±1 std')
plt.axvline(best_alpha_ridge, color='red', linestyle='--',
            label=f'Best α = {best_alpha_ridge:.2f}')
plt.xlabel('Alpha (log scale)')
plt.ylabel('R² (5-Fold CV)')
plt.title('Ridge: Validation Curve — Alpha Tuning via K-Fold')
plt.legend()
plt.tight_layout()
plt.show()

## 🟠 Section 2 — Lasso Regression (Automatic Feature Selection)

In [ ]:
# LassoCV runs K-Fold internally across a range of alphas
lasso_cv = LassoCV(alphas=alphas, cv=kf, max_iter=10000)
lasso_cv.fit(X_tr_poly, y_tr)

best_alpha_lasso = lasso_cv.alpha_
print(f'Lasso best alpha (5-Fold CV): {best_alpha_lasso:.4f}')

r2_lasso_train = r2_score(y_tr, lasso_cv.predict(X_tr_poly))
r2_lasso_test  = r2_score(y_te, lasso_cv.predict(X_te_poly))

# Count zero coefficients
n_zero   = np.sum(np.abs(lasso_cv.coef_) < 1e-10)
n_nonzero= np.sum(np.abs(lasso_cv.coef_) >= 1e-10)

print(f'Lasso — Train R²: {r2_lasso_train:.4f}  Test R²: {r2_lasso_test:.4f}')
print(f'Features zeroed out: {n_zero} / {len(lasso_cv.coef_)}')
print(f'Features retained:   {n_nonzero}')

## 📈 Section 3 — Regularisation Paths

In [ ]:
# ── Ridge regularisation path ─────────────────────────────────────────────────
# Shows how coefficients shrink (but never reach 0) as alpha increases

alphas_path = np.logspace(-2, 4, 100)

ridge_coefs = []
lasso_coefs = []

for a in alphas_path:
    r = Ridge(alpha=a).fit(X_tr_poly, y_tr)
    l = Lasso(alpha=a, max_iter=5000).fit(X_tr_poly, y_tr)
    ridge_coefs.append(r.coef_[:8])   # show first 8 coefficients
    lasso_coefs.append(l.coef_[:8])

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i in range(8):
    axes[0].semilogx(alphas_path, ridge_coefs[:, i])
    axes[1].semilogx(alphas_path, lasso_coefs[:, i])

for ax, title in zip(axes, ['Ridge Path (L2)', 'Lasso Path (L1)']):
    ax.axvline(best_alpha_ridge if 'Ridge' in title else best_alpha_lasso,
               color='red', linestyle='--', label='Best α')
    ax.set_xlabel('Alpha')
    ax.set_ylabel('Coefficient')
    ax.set_title(title)
    ax.legend()

plt.suptitle('Regularisation Paths — Ridge vs Lasso', fontsize=14)
plt.tight_layout()
plt.show()

print('💡 Ridge: coefficients shrink smoothly, never reach zero.')
print('   Lasso: coefficients hit zero and stay there — built-in feature selection!')

## ⚡ Section 4 — Elastic Net

In [ ]:
from sklearn.linear_model import ElasticNetCV

enet_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    alphas=alphas,
    cv=kf,
    max_iter=10000
)
enet_cv.fit(X_tr_poly, y_tr)

r2_enet = r2_score(y_te, enet_cv.predict(X_te_poly))
print(f'ElasticNet — Best alpha: {enet_cv.alpha_:.4f}  l1_ratio: {enet_cv.l1_ratio_:.2f}')
print(f'ElasticNet — Test R²: {r2_enet:.4f}')

## 📊 Section 5 — Performance Summary

In [ ]:
summary = pd.DataFrame([
    {'Model': 'OLS (no regularisation)', 'Train R²': r2_train, 'Test R²': r2_test,
     'Overfit Gap': r2_train-r2_test, 'Non-zero Coefs': X_tr_poly.shape[1]},
    {'Model': f'Ridge (α={best_alpha_ridge:.1f})',
     'Train R²': r2_ridge_train, 'Test R²': r2_ridge_test,
     'Overfit Gap': r2_ridge_train-r2_ridge_test, 'Non-zero Coefs': X_tr_poly.shape[1]},
    {'Model': f'Lasso (α={best_alpha_lasso:.3f})',
     'Train R²': r2_lasso_train, 'Test R²': r2_lasso_test,
     'Overfit Gap': r2_lasso_train-r2_lasso_test, 'Non-zero Coefs': n_nonzero},
    {'Model': f'ElasticNet',
     'Train R²': None, 'Test R²': r2_enet,
     'Overfit Gap': None, 'Non-zero Coefs': None},
]).round(4)

print(summary.to_string(index=False))

# Bar chart
models = ['OLS', f'Ridge α={best_alpha_ridge:.0f}', f'Lasso α={best_alpha_lasso:.3f}', 'ElasticNet']
test_scores = [r2_test, r2_ridge_test, r2_lasso_test, r2_enet]

plt.figure(figsize=(9, 4))
colors = ['tomato', 'steelblue', 'limegreen', 'orange']
plt.bar(models, test_scores, color=colors, edgecolor='white')
plt.ylabel('Test R²')
plt.title('Model Comparison: Regularisation Effect on Test R²')
plt.ylim(0, 1)
for i, (m, v) in enumerate(zip(models, test_scores)):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 🎯 Summary

| Regulariser | Penalty | Effect | K-Fold role |
|---|---|---|---|
| Ridge (L2) | α‖β‖² | Shrinks all coefs | Tune α value |
| Lasso (L1) | α‖β‖₁ | Zeros out coefs | Tune α value |
| Elastic Net | α₁‖β‖₁ + α₂‖β‖² | Hybrid | Tune both α and l1_ratio |

**K-Fold Insights:**
- K-Fold gave a **statistically robust estimate** of the best alpha
- Without K-Fold, you risk tuning to lucky test-set performance
- Regularisation **reduced the overfit gap** while maintaining good test performance

---

**Next:** [07_linear_regression_tensorflow.ipynb](07_linear_regression_tensorflow.ipynb)